# Fine-tune FunctionGemma 270M for Edge Assistant voice commands

Runs on a free **Colab T4**: *Runtime → Change runtime type → T4 GPU*. About 15–25 minutes end to end.

1. Clones the repo (dataset, function schema, scoring code)
2. Scores the **base** model on `data/eval_seed.jsonl` (66 held-out commands, incl. real phone transcripts)
3. Full fine-tune on `data/train.jsonl` (loss on the answer only)
4. Scores again, exports **GGUF q8_0**, downloads it

Before step 2 you need a Hugging Face token, and to accept the Gemma license on
https://huggingface.co/google/functiongemma-270m-it

In [ ]:
import os
%cd /content
if os.path.isdir("edge_ai/.git"):
    !git -C edge_ai pull -q --ff-only
else:
    !rm -rf edge_ai && git clone -q https://github.com/surajpatil4899/edge_ai.git
%cd /content/edge_ai/finetune
!pip install -q -U transformers trl datasets accelerate

In [ ]:
from huggingface_hub import login
login()  # paste a token with read access; the Gemma license must be accepted on the model page

In [ ]:
import json, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from evaluate import score, summarize          # same scoring as finetune/evaluate.py
from generate_data import SYSTEM

MODEL = "google/functiongemma-270m-it"
tok = AutoTokenizer.from_pretrained(MODEL)
tools = json.load(open("functions.json"))

def to_prompt_completion(row):
    msgs = row["messages"]
    prompt = tok.apply_chat_template(msgs[:-1], tools=tools, add_generation_prompt=True, tokenize=False)
    full = tok.apply_chat_template(msgs, tools=tools, tokenize=False)
    assert full.startswith(prompt), "chat template is not prefix-stable"
    return {"prompt": prompt, "completion": full[len(prompt):]}

load = lambda path: [json.loads(l) for l in open(path)]
train = Dataset.from_list([to_prompt_completion(r) for r in load("data/train.jsonl")])
val = Dataset.from_list([to_prompt_completion(r) for r in load("data/val.jsonl")])
eval_rows = load("data/eval_seed.jsonl")
print(len(train), "train /", len(val), "val /", len(eval_rows), "eval")
print("prompt tokens:", len(tok(train[0]["prompt"])["input_ids"]))
for i in range(3):
    print(repr(train[i]["prompt"][-80:]), "→", repr(train[i]["completion"]))

In [ ]:
@torch.no_grad()
def run_eval(model, rows, show_errors=True):
    model.eval()
    results = []
    for r in rows:
        msgs = [{"role": "developer", "content": SYSTEM}, {"role": "user", "content": r["text"]}]
        inputs = tok.apply_chat_template(msgs, tools=tools, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
        raw = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False).split("<end_function_call>")[0]
        res = score(r, raw)
        results.append(res)
        if show_errors and not res["exact"]:
            print(f"  ✗ {r['text']!r:55.55} {'; '.join(res['errors'])}")
    print(summarize(results))
    return results

base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32, attn_implementation="eager").to("cuda")
base_results = run_eval(base, eval_rows)
del base; torch.cuda.empty_cache()

In [ ]:
import inspect, math
from trl import SFTConfig, SFTTrainer

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32, attn_implementation="eager")
EPOCHS, BATCH, ACCUM = 3, 8, 2
total_steps = math.ceil(len(train) / (BATCH * ACCUM)) * EPOCHS
settings = dict(
    output_dir="out",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_steps=max(1, int(0.05 * total_steps)),   # an int works across transformers versions
    weight_decay=0.01,
    max_length=1536,
    completion_only_loss=True,      # learn the answer, not the 1000-token tool declarations
    fp16=True,                      # T4 has no bf16; fp32 master weights + fp16 compute
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    report_to="none",
)
# Library versions rename/remove arguments; skip unknown ones instead of failing.
accepted = set(inspect.signature(SFTConfig.__init__).parameters)
for k in [k for k in settings if k not in accepted]:
    print(f"note: this TRL/transformers version has no '{k}', skipping it")
    settings.pop(k)
config = SFTConfig(**settings)
print(f"{total_steps} optimizer steps")
trainer = SFTTrainer(model=model, args=config, train_dataset=train, eval_dataset=val, processing_class=tok)
trainer.train()

In [ ]:
tuned_results = run_eval(trainer.model, eval_rows)
json.dump({"base": base_results, "tuned": tuned_results}, open("eval_results.json", "w"), indent=1)

In [ ]:
from huggingface_hub import snapshot_download

OUT = "functiongemma-edge"
trainer.model.to(torch.float32).save_pretrained(OUT)
# Fine-tuning doesn't change the tokenizer. Newer transformers' save_pretrained omits tokenizer.model,
# which llama.cpp's Gemma converter needs, so ship the original tokenizer files instead.
snapshot_download(MODEL, local_dir=OUT, allow_patterns=[
    "tokenizer.model", "tokenizer.json", "tokenizer_config.json",
    "special_tokens_map.json", "added_tokens.json", "chat_template.jinja"])

# Only what the converter needs: its requirements file pins old huggingface-hub/protobuf and
# downgrades Colab's packages. The converter imports its own gguf package from the repo.
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q sentencepiece
!python llama.cpp/convert_hf_to_gguf.py {OUT} --outtype q8_0 --outfile {OUT}-q8_0.gguf 2>&1 | tail -3
!ls -lh {OUT}-q8_0.gguf

print("BASE\n" + summarize(base_results))
print("\nFINE-TUNED\n" + summarize(tuned_results))

In [ ]:
from google.colab import files
files.download(f"{OUT}-q8_0.gguf")
files.download("eval_results.json")